# Operational Workflow: Healthcare Data Documentation with ADE

This notebook demonstrates the operational workflow for using the Agent-Driven Evaluation (ADE) system to document healthcare data. 
It utilizes the refactored `ade` package for modularity and robustness.

In [ ]:
# Setup and Imports
import sys
import os
import getpass

# Ensure package is in path if running locally
sys.path.append(os.path.abspath('.'))

# --- Credentials Setup ---
if "GOOGLE_API_KEY" not in os.environ:
    print("Google API Key not found in environment variables.")
    # Check if we are running in Colab or Kaggle and try to retrieve secrets
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
        print("Loaded key from Colab userdata.")
    except ImportError:
        # Fallback to interactive input
        print("Please provide your Google API Key (input will be hidden):")
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("API Key: ")

from ade import (
    API_CONFIG,
    EnhancedDatabaseManager,
    Orchestrator,
    ReviewQueueManager
)
from ade.ui import (
    HITLReviewDashboard,
    DocumentUploader,
    BatchOperationsWidget,
    ClarificationWidget,
    ExportWidget
)
import ipywidgets as widgets
from IPython.display import display

# Use a persistent database file
DB_PATH = 'healthcare_docs.db'

print("ADE Package Imports Successful")

## 1. System Initialization
Initialize the database manager and orchestrator.

In [ ]:
# Initialize Database
db_manager = EnhancedDatabaseManager(DB_PATH)
db_manager.connect()
db_manager.initialize_schema()

# Initialize Orchestrator
orchestrator = Orchestrator(db_manager)

# Initialize Review Queue
review_queue = ReviewQueueManager(db_manager)

print("System Initialized")

## 2. Upload Data
Upload your data dictionary files (CSV, Excel) for processing.

In [ ]:
uploader = DocumentUploader()
display(uploader.create_widget())

## 3. Process Data
Process the uploaded data through the agent pipeline.

In [ ]:
# Check if data is uploaded
if uploader.uploaded_data is not None:
    # Convert to CSV string/format expected by orchestrator
    data_str = uploader.uploaded_data.to_csv(index=False)
    source_file = "uploaded_file.csv" # In real usage, get filename from uploader
    
    # Start Processing Job
    # Note: process_data_dictionary handles transaction safety
    job_id = orchestrator.process_data_dictionary(
        source_data=data_str,
        source_file=source_file,
        auto_approve=False
    )
    
    print(f"Job Started: {job_id}")
else:
    print("Please upload a file in the previous step.")

## 4. HITL Review
Review the generated documentation, approve, reject, or edit items.

In [ ]:
if 'job_id' in locals():
    dashboard = HITLReviewDashboard(review_queue)
    display(dashboard.create_widget(job_id))
else:
    print("No active job. Run the processing step first.")

## 5. Batch Operations
Perform bulk actions (Approve All/Reject All) if needed.

In [ ]:
if 'job_id' in locals():
    batch_ops = BatchOperationsWidget(review_queue)
    display(batch_ops.create_widget(job_id))
else:
    print("No active job.")

## 6. Export Documentation
Export the approved documentation to a Markdown file.

In [ ]:
if 'job_id' in locals():
    export_widget = ExportWidget(orchestrator.assembler, review_queue)
    display(export_widget.create_widget(job_id))
else:
    print("No active job.")